# EDA — Amazon Reviews Multilingual (es)

**Módulo 2 — Análisis de sentimiento**

Análisis exploratorio del dataset Amazon Reviews en español antes de entrenar.
Objetivo: entender la distribución de sentimientos, calidad del texto y decidir estrategia de balanceo.

In [ ]:
# Montar Drive y configurar entorno (Colab)
from google.colab import drive
drive.mount('/content/drive')

!pip install -q polars datasets wordcloud seaborn nltk

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import nltk
from collections import Counter
from wordcloud import WordCloud
from datasets import load_dataset

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

SEED = 42
LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL_COLORS = {'negative': '#e74c3c', 'neutral': '#f39c12', 'positive': '#2ecc71'}
STOPWORDS_ES = set(stopwords.words('spanish'))

sns.set_theme(style='whitegrid', palette='muted')
print('Entorno listo.')

In [ ]:
import polars as pl
from datasets import load_dataset

# amazon_reviews_multi usa un script legacy incompatible con datasets >= 4.0.
# mteb/amazon_reviews_multi es el mismo dataset en formato parquet nativo.
# Diferencia de esquema: text → review_body, label (0-4) → stars (1-5).
print('Cargando dataset...')
raw = load_dataset("mteb/amazon_reviews_multi", "es")

def to_polars(split: str) -> pl.DataFrame:
    return (
        pl.from_arrow(raw[split].data.table)
        .select(["text", "label"])
        .rename({"text": "review_body", "label": "stars"})
        .with_columns(pl.col("stars") + 1)
    )

train_df = to_polars("train")
test_df  = to_polars("test")
val_df   = to_polars("validation")

df = pl.concat([train_df, test_df, val_df])
print(f"Total reseñas: {len(df):,}  |  Train: {len(train_df):,}  Test: {len(test_df):,}  Val: {len(val_df):,}")
df.head(5)

## 1. Mapeo de ratings → sentimiento

In [ ]:
df = df.with_columns(
    pl.when(pl.col('stars') <= 2).then(0)
      .when(pl.col('stars') == 3).then(1)
      .otherwise(2)
      .alias('label')
).with_columns(
    pl.col('label').replace_strict(LABEL_MAP, return_dtype=pl.String).alias('sentiment')
)

print(df.select(['stars', 'label', 'sentiment', 'review_body']).head(8))

## 2. Distribución de ratings y sentimientos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Distribución de ratings (1-5) ---
rating_counts = (
    df.group_by('stars').agg(pl.len().alias('count'))
      .sort('stars')
)
axes[0].bar(
    rating_counts['stars'].to_list(),
    rating_counts['count'].to_list(),
    color='steelblue', edgecolor='white'
)
axes[0].set_title('Distribución de ratings (1–5)', fontsize=13)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Cantidad de reseñas')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(axes[0].patches, rating_counts['count'].to_list()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

# --- Distribución de sentimientos ---
sent_counts = (
    df.group_by('sentiment').agg(pl.len().alias('count'))
      .sort('count', descending=True)
)
colors = [LABEL_COLORS[s] for s in sent_counts['sentiment'].to_list()]
bars = axes[1].bar(
    sent_counts['sentiment'].to_list(),
    sent_counts['count'].to_list(),
    color=colors, edgecolor='white'
)
axes[1].set_title('Distribución de sentimientos', fontsize=13)
axes[1].set_xlabel('Sentimiento')
axes[1].set_ylabel('Cantidad de reseñas')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
total = len(df)
for bar, val in zip(bars, sent_counts['count'].to_list()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,} ({val/total:.1%})', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('dist_ratings_sentimientos.png', dpi=150, bbox_inches='tight')
plt.show()

# Ratio de desbalance
counts_dict = dict(zip(sent_counts['sentiment'].to_list(), sent_counts['count'].to_list()))
ratio = max(counts_dict.values()) / min(counts_dict.values())
print(f'\nRatio mayoritaria / minoritaria: {ratio:.2f}x')
print('Advertencia: ratio > 5 requiere oversampling o undersampling' if ratio > 5 else 'OK: class_weight=balanced es suficiente')

## 3. Longitud de reseñas

In [ ]:
df = df.with_columns([
    pl.col('review_body').str.len_chars().alias('char_len'),
    pl.col('review_body').str.split(' ').list.len().alias('word_len'),
])

# Estadísticas por clase
stats = (
    df.group_by('sentiment').agg([
        pl.col('word_len').mean().alias('word_mean'),
        pl.col('word_len').median().alias('word_median'),
        pl.col('word_len').quantile(0.95).alias('word_p95'),
        pl.col('char_len').mean().alias('char_mean'),
    ])
)
print('Estadísticas de longitud por clase:')
print(stats.sort('sentiment'))

# Histogramas por clase
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, label in zip(axes, ['negative', 'neutral', 'positive']):
    vals = df.filter(pl.col('sentiment') == label)['word_len'].to_list()
    ax.hist(vals, bins=60, color=LABEL_COLORS[label], alpha=0.85, edgecolor='white')
    ax.set_title(f'{label.capitalize()} — longitud (palabras)', fontsize=11)
    ax.set_xlabel('Palabras')
    ax.set_ylabel('Frecuencia')
    ax.axvline(256, color='black', linestyle='--', linewidth=1, label='max BETO (≈256)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('longitud_resenas.png', dpi=150, bbox_inches='tight')
plt.show()

# Porcentaje de reseñas que exceden 256 tokens (proxy)
over_256 = df.filter(pl.col('word_len') > 256).shape[0]
print(f'\nReseñas con > 256 palabras (se truncan en BETO): {over_256:,} ({over_256/len(df):.2%})')

## 4. Palabras más frecuentes por sentimiento

In [ ]:
import re
import unicodedata

def tokenize(text: str) -> list[str]:
    text = unicodedata.normalize('NFKD', str(text)).lower()
    text = re.sub(r'[^\w\sáéíóúüñ]', ' ', text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS_ES and len(t) > 2]

TOP_N = 20
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

word_freqs = {}
for ax, label in zip(axes, ['negative', 'neutral', 'positive']):
    texts = df.filter(pl.col('sentiment') == label)['review_body'].to_list()
    all_tokens = []
    for t in texts:
        all_tokens.extend(tokenize(t))
    freq = Counter(all_tokens).most_common(TOP_N)
    word_freqs[label] = dict(Counter(all_tokens))
    words, counts = zip(*freq)
    ax.barh(words[::-1], counts[::-1], color=LABEL_COLORS[label], alpha=0.85)
    ax.set_title(f'Top {TOP_N} palabras — {label.capitalize()}', fontsize=11)
    ax.set_xlabel('Frecuencia')

plt.tight_layout()
plt.savefig('top_palabras.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Nubes de palabras

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, label in zip(axes, ['negative', 'neutral', 'positive']):
    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap='RdYlGn' if label == 'positive' else ('Reds' if label == 'negative' else 'Oranges'),
        max_words=150,
    ).generate_from_frequencies(word_freqs[label])
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'WordCloud — {label.capitalize()}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Detección de reseñas problemáticas

In [ ]:
n_total = len(df)

# Vacías o solo espacios
empty = df.filter(pl.col('review_body').str.strip_chars().str.len_chars() == 0)

# Duplicados exactos
duplicates = df.filter(pl.col('review_body').is_duplicated())

# Reseñas muy cortas (< 5 palabras — probable ruido)
too_short = df.filter(pl.col('word_len') < 5)

# Encoding roto / HTML (presencia de & ; < > o \x)
has_html = df.filter(
    pl.col('review_body').str.contains(r'&[a-z]+;|<[^>]+>|\\x[0-9a-f]{2}')
)

print(f'Total reseñas:              {n_total:>8,}')
print(f'Vacías o solo espacios:     {len(empty):>8,}  ({len(empty)/n_total:.3%})')
print(f'Duplicados exactos:         {len(duplicates):>8,}  ({len(duplicates)/n_total:.3%})')
print(f'Muy cortas (< 5 palabras):  {len(too_short):>8,}  ({len(too_short)/n_total:.3%})')
print(f'Con caracteres HTML/broken: {len(has_html):>8,}  ({len(has_html)/n_total:.3%})')

# Ejemplos
if len(too_short) > 0:
    print('\nEjemplos de reseñas muy cortas:')
    print(too_short.select('review_body').head(5))

## 7. Decisión de balanceo

In [ ]:
sent_dist = (
    df.group_by('sentiment').agg(pl.len().alias('count'))
      .sort('count', descending=True)
)
counts_dict = dict(zip(sent_dist['sentiment'].to_list(), sent_dist['count'].to_list()))
ratio = max(counts_dict.values()) / min(counts_dict.values())

print('=== DECISIÓN DE BALANCEO ===')
for label, cnt in sorted(counts_dict.items()):
    print(f'  {label:<10}: {cnt:>7,}  ({cnt/n_total:.1%})')
print(f'\n  Ratio mayor/menor: {ratio:.2f}x')

if ratio <= 3:
    decision = 'class_weight="balanced"'
    justification = 'Desbalance moderado (≤ 3x). El parámetro class_weight=balanced en RandomForest es suficiente — ya está configurado en baseline_model.py.'
elif ratio <= 5:
    decision = 'class_weight="balanced" + monitoreo por clase'
    justification = 'Desbalance notable (3–5x). Usar class_weight=balanced y reportar F1 por clase para detectar degradación en la clase minoritaria.'
else:
    decision = 'Undersampling de la clase mayoritaria'
    justification = 'Desbalance severo (> 5x). Undersampling al 3x de la clase minoritaria antes de entrenar.'

print(f'\n  DECISIÓN: {decision}')
print(f'  JUSTIFICACIÓN: {justification}')

## 8. Conclusiones

In [ ]:
avg_words = df['word_len'].mean()
median_words = df['word_len'].median()
pct_over_256 = df.filter(pl.col('word_len') > 256).shape[0] / n_total

print('=' * 55)
print('CONCLUSIONES DEL EDA — Amazon Reviews (es)')
print('=' * 55)
print(f'Total de reseñas analizadas : {n_total:,}')
print()
print('Distribución de sentimientos:')
for label, cnt in sorted(counts_dict.items()):
    print(f'  {label:<10}: {cnt:>7,}  ({cnt/n_total:.1%})')
print(f'  Ratio mayor/menor: {ratio:.2f}x')
print()
print(f'Longitud promedio   : {avg_words:.1f} palabras')
print(f'Longitud mediana    : {median_words:.0f} palabras')
print(f'Reseñas > 256 words : {pct_over_256:.2%}  (se truncan en BETO — aceptable)')
print()
print('Calidad del dataset:')
print(f'  Vacías            : {len(empty):,}')
print(f'  Duplicados        : {len(duplicates):,}')
print(f'  Muy cortas (<5w)  : {len(too_short):,}')
print(f'  HTML/encoding roto: {len(has_html):,}')
print()
print(f'Técnica de balanceo elegida: {decision}')
print()
print('Próximo paso: notebooks/02_sentiment_baseline_randomforest.ipynb')